In [1]:
import pandas as pd 
import numpy as np 

In [3]:
df = pd.read_csv("D:/UIDAI HACKATHON/UIDAI-Hackathon/data/processed/Aadhaar_daily.csv")
df.head()

,date,state,district,pincode,age_0_5,age_5_17,age_18_greater,demo_age_5_17,demo_age_17_,bio_age_5_17,bio_age_17_
0,01-01-2026,Maharashtra,Thane,400601,14.0,3.0,0.0,3.0,31.0,48.0,67.0
1,01-03-2025,Maharashtra,Thane,400601,0.0,0.0,0.0,99.0,1058.0,0.0,0.0
2,01-04-2025,Maharashtra,Thane,400601,0.0,0.0,0.0,104.0,783.0,393.0,831.0
3,01-05-2025,Maharashtra,Thane,400601,80.0,30.0,14.0,0.0,0.0,453.0,851.0
4,01-07-2025,Maharashtra,Thane,400601,0.0,0.0,0.0,0.0,0.0,14.0,32.0


### Changing the dates into month-year combinations

In [6]:
import datetime
df['date'] = pd.to_datetime(
    df['date'],
    dayfirst=True,
    errors='coerce'
)

# Safety check
assert df['date'].isna().sum() == 0, "Some dates failed to parse"


In [7]:
df['month'] = (
    pd.to_datetime(df['date'], dayfirst=True, errors='coerce')
      .dt.to_period('M')
      .dt.to_timestamp()
)


### Aggregating updates + enrollments to make data monthly

In [9]:
monthly = (
    df
    .groupby(['state', 'district', 'pincode', 'month'], as_index=False)
    .agg({
        'age_0_5': 'sum',
        'age_5_17': 'sum',
        'age_18_greater': 'sum',
        'demo_age_5_17': 'sum',
        'demo_age_17_': 'sum',
        'bio_age_5_17': 'sum',
        'bio_age_17_': 'sum'
    })
)


In [10]:
assert monthly.duplicated(
    subset=['pincode', 'month']
).sum() == 0


In [14]:
monthly.head()

,state,district,pincode,month,age_0_5,age_5_17,age_18_greater,demo_age_5_17,demo_age_17_,bio_age_5_17,bio_age_17_
0,Maharashtra,Thane,400601,2025-03-01,0.0,0.0,0.0,99.0,1058.0,0.0,0.0
1,Maharashtra,Thane,400601,2025-04-01,0.0,0.0,0.0,104.0,783.0,393.0,831.0
2,Maharashtra,Thane,400601,2025-05-01,80.0,30.0,14.0,0.0,0.0,575.0,1122.0
3,Maharashtra,Thane,400601,2025-06-01,53.0,33.0,1.0,0.0,0.0,165.0,670.0
4,Maharashtra,Thane,400601,2025-07-01,0.0,0.0,0.0,0.0,0.0,407.0,912.0


In [15]:
monthly['month'] = monthly['month'].dt.strftime('%Y-%m')


In [16]:
monthly.head()

,state,district,pincode,month,age_0_5,age_5_17,age_18_greater,demo_age_5_17,demo_age_17_,bio_age_5_17,bio_age_17_
0,Maharashtra,Thane,400601,2025-03,0.0,0.0,0.0,99.0,1058.0,0.0,0.0
1,Maharashtra,Thane,400601,2025-04,0.0,0.0,0.0,104.0,783.0,393.0,831.0
2,Maharashtra,Thane,400601,2025-05,80.0,30.0,14.0,0.0,0.0,575.0,1122.0
3,Maharashtra,Thane,400601,2025-06,53.0,33.0,1.0,0.0,0.0,165.0,670.0
4,Maharashtra,Thane,400601,2025-07,0.0,0.0,0.0,0.0,0.0,407.0,912.0


In [18]:
monthly.shape

(982, 11)

In [19]:
monthly.to_csv(
    'D:/UIDAI HACKATHON/UIDAI-Hackathon/data/processed/monthly_pin_raw.csv',
    index=False
)
